# 01 - Data processing (Group Sprint 2: freeze-frame xG)

**Goal:** turn the four raw CSVs into two model-ready tables, the **shot table** (1 row per shot) and the **player table** (1 row per player per shot, including the shooter), ready for the set model and the graph model.

**No modelling happens here.** Every removal, addition and transformation is written to a processing log (with the reasoning) which is shown at the end and saved next to the outputs, so we can use it in the reflections.

Contents
1. Setup and config
2. Load raw data
3. Data-quality checks
4. Filter shots
5. Shot table: select columns, clean, geometry features
6. Player table: frames, shooter row, geometry features
7. Train / validation / test split (by match)
8. One-hot encoding (shot table)
9. Standardisation (fit on train only)
10. Final columns and feature lists
11. Validation checks
12. Save outputs
13. Processing log

In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 170)

# ------------------------------- CONFIG --------------------------------
PROJECT_DIR = None          # e.g. Path(r"C:/.../Data Science Toolbox/Group Sprint 2"); None = auto-detect
RANDOM_SEED = 42
EXCLUDE_PENALTIES = True    # penalties have almost no useful freeze-frame information (~77% scored)
RARE_MIN = 50               # categories with fewer than this many TRAIN shots are grouped into "Other"
SPLIT_FRACS = {"train": 0.70, "val": 0.15, "test": 0.15}

# StatsBomb pitch: 120 x 80, every team attacks towards x = 120. Goal centre (120, 40), posts at y = 36 and 44.
PITCH_X, PITCH_Y = 120.0, 80.0
GOAL_X, GOAL_Y = 120.0, 40.0
POST_LO, POST_HI = 36.0, 44.0
# -----------------------------------------------------------------------

def find_project_dir():
    start = Path.cwd().resolve()
    for base in [start, *start.parents]:
        for rel in ["", "Group Sprint 2", "Data Science Toolbox/Group Sprint 2"]:
            p = base / rel
            if (p / "data" / "raw").is_dir():
                return p
    raise FileNotFoundError("Could not find a 'data/raw' folder. Set PROJECT_DIR in the config cell.")

PROJECT_DIR = Path(PROJECT_DIR) if PROJECT_DIR else find_project_dir()
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROC_DIR = PROJECT_DIR / "data" / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)
print("Project dir  :", PROJECT_DIR)
print("Raw data     :", RAW_DIR)
print("Output folder:", PROC_DIR)

# ------------------------------ PROCESSING LOG -------------------------
LOG = []
def log(step, action, item, detail, reason):
    """Record one processing decision. Everything shown in the final log comes from here."""
    LOG.append({"n": len(LOG) + 1, "step": step, "action": action, "item": item,
                "detail": detail, "reason": reason})
    print(f"[{len(LOG):02d}] {step} | {action} | {item}: {detail}")

# ---------------------------- GEOMETRY HELPERS -------------------------
def subtended_angle(x, y):
    """Angle (radians) at point (x, y) between the two goalposts. Larger = more of the goal 'visible'."""
    ax = GOAL_X - x
    a1 = POST_LO - y
    a2 = POST_HI - y
    dot = ax * ax + a1 * a2
    cross = ax * a2 - a1 * ax
    return np.arctan2(np.abs(cross), dot)

def dist_to_segment(px, py, sx, sy, gx, gy):
    """Distance from point P to the segment S->G (shooter to goal centre)."""
    vx, vy = gx - sx, gy - sy
    L2 = vx * vx + vy * vy
    safe = np.where(L2 > 1e-12, L2, 1.0)
    t = np.where(L2 > 1e-12, ((px - sx) * vx + (py - sy) * vy) / safe, 0.0)
    t = np.clip(t, 0.0, 1.0)
    return np.hypot(px - (sx + t * vx), py - (sy + t * vy))

def in_triangle(px, py, sx, sy):
    """True if P lies inside the triangle shooter -> left post -> right post (boundary counts as inside)."""
    ax_, ay_, bx_, by_ = GOAL_X, POST_LO, GOAL_X, POST_HI
    def cr(x1, y1, x2, y2, x3, y3):
        return (x2 - x1) * (y3 - y1) - (y2 - y1) * (x3 - x1)
    d1 = cr(sx, sy, ax_, ay_, px, py)
    d2 = cr(ax_, ay_, bx_, by_, px, py)
    d3 = cr(bx_, by_, sx, sy, px, py)
    neg = (d1 < 0) | (d2 < 0) | (d3 < 0)
    pos = (d1 > 0) | (d2 > 0) | (d3 > 0)
    area = cr(sx, sy, ax_, ay_, bx_, by_)          # 0 if the shooter is on the goal line (degenerate triangle)
    return (~(neg & pos)) & (np.abs(area) > 1e-9)

Project dir  : /home/jovyan/Data Science Toolbox/Group Sprint 2
Raw data     : /home/jovyan/Data Science Toolbox/Group Sprint 2/data/raw
Output folder: /home/jovyan/Data Science Toolbox/Group Sprint 2/data/processed


## 2. Load raw data

In [2]:
STEP = "2. Load"
def to_bool(s):
    return s.astype(str).str.strip().str.upper().eq("TRUE")

shots   = pd.read_csv(RAW_DIR / "Neural Network_ xG - shots.csv")
ff      = pd.read_csv(RAW_DIR / "Neural Network_ xG - freeze_frames.csv")
matches = pd.read_csv(RAW_DIR / "Neural Network_ xG - matches.csv")
summary = pd.read_csv(RAW_DIR / "Neural Network_ xG - summary.csv")

for c in ["under_pressure", "first_time", "is_goal", "has_freeze_frame", "season_complete"]:
    shots[c] = to_bool(shots[c])
for c in ["teammate", "is_keeper"]:
    ff[c] = to_bool(ff[c])
matches["season_complete"] = to_bool(matches["season_complete"])
summary = summary.dropna(subset=["league"]).copy()      # last row of the summary sheet is a totals row

log(STEP, "LOAD", "shots.csv", f"{shots.shape[0]:,} rows x {shots.shape[1]} cols", "Main shot-level data (one row per shot).")
log(STEP, "LOAD", "freeze_frames.csv", f"{ff.shape[0]:,} rows x {ff.shape[1]} cols", "One row per visible player per shot.")
log(STEP, "LOAD", "matches.csv", f"{matches.shape[0]:,} rows", "Used only for integrity checks and context; not a model input.")
log(STEP, "LOAD", "summary.csv", f"{summary.shape[0]} league-seasons", "Used only to reconcile our counts with the source; not a model input.")
log(STEP, "TRANSFORM", "boolean columns", "TRUE/FALSE text parsed to bool", "Make sure flags are real booleans before converting to 0/1 later.")
shots.head(3)

[01] 2. Load | LOAD | shots.csv: 54,153 rows x 30 cols
[02] 2. Load | LOAD | freeze_frames.csv: 685,059 rows x 7 cols
[03] 2. Load | LOAD | matches.csv: 2,169 rows
[04] 2. Load | LOAD | summary.csv: 27 league-seasons
[05] 2. Load | TRANSFORM | boolean columns: TRUE/FALSE text parsed to bool


,shot_id,match_id,minute,second,period,team,player,play_pattern,under_pressure,location_x,location_y,shot_type,body_part,technique,first_time,outcome,is_goal,statsbomb_xg,league,season,match_date,season_complete,has_freeze_frame,end_x,end_y,end_z,ff_n_players,ff_n_teammates,ff_n_opponents,ff_has_keeper
0,c28405cb-0742-43d0-80b9-7ff486710e0c,68321,3,31,1,Barcelona,Ronaldo de Assis Moreira,From Corner,False,119.7,0.7,Corner,Right Foot,Normal,False,Post,False,0.00018,La Liga,2005/2006,2005-12-11,False,True,120.0,35.7,2.7,15,5,10,True
1,538f0ed8-576f-420c-adab-09e8c97c34b4,3754162,1,40,1,AFC Bournemouth,Junior Stanislas,From Corner,False,120.2,1.0,Corner,Right Foot,Normal,False,Goal,True,0.00018,Premier League,2015/2016,2015-12-12,True,True,120.0,43.4,2.0,17,6,11,True
2,547c60b4-eee4-4430-a7ab-ad088531590a,3825889,66,0,2,Real Betis,Joaquín Sánchez Rodríguez,From Corner,False,119.0,0.7,Corner,Right Foot,Normal,False,Saved,False,0.00018,La Liga,2015/2016,2016-04-16,True,True,119.9,36.9,2.0,17,7,10,True


## 3. Data-quality checks (nothing is changed here, only logged)

In [3]:
STEP = "3. Checks"

dups = int(shots.shot_id.duplicated().sum())
log(STEP, "CHECK", "duplicate shot_id", f"{dups} duplicates", "shot_id must be a unique key for the join between the two tables.")
assert dups == 0

miss = shots.isna().sum()
miss = miss[miss > 0].to_dict()
log(STEP, "CHECK", "missing values in shots", str(miss) if miss else "none",
    "Only end_z has gaps; it is a post-shot column that we remove anyway (see step 5).")
miss_ff = ff.isna().sum()
miss_ff = miss_ff[miss_ff > 0].to_dict()
log(STEP, "CHECK", "missing values in freeze_frames", str(miss_ff) if miss_ff else "none", "No imputation needed if empty.")

by = shots.groupby(["league", "season"]).agg(shots_n=("shot_id", "size"), goals_n=("is_goal", "sum"),
                                              ff_n=("has_freeze_frame", "sum")).reset_index()
chk = summary.merge(by, on=["league", "season"], how="outer")
bad = chk[(chk.shots != chk.shots_n) | (chk.goals != chk.goals_n) | (chk.with_freeze_frame != chk.ff_n)]
log(STEP, "CHECK", "reconcile with summary.csv", f"{len(bad)} league-seasons disagree out of {len(chk)}",
    "Confirms the shots file matches the source counts (shots, goals, shots with freeze frames).")

n_unmatched = len(set(shots.match_id) - set(matches.match_id))
log(STEP, "CHECK", "shots whose match_id is not in matches.csv", f"{n_unmatched} match_ids", "Referential integrity between tables.")

orphans = len(set(ff.shot_id) - set(shots.shot_id))
flag_no_rows = int((shots.has_freeze_frame & ~shots.shot_id.isin(set(ff.shot_id))).sum())
rows_no_flag = int((~shots.has_freeze_frame & shots.shot_id.isin(set(ff.shot_id))).sum())
log(STEP, "CHECK", "freeze-frame consistency",
    f"{orphans} frame shot_ids not in shots; {flag_no_rows} shots flagged as having a frame but no rows; {rows_no_flag} shots with rows but flag False",
    "Decides whether has_freeze_frame can be trusted as the filter (step 4 uses both).")

oob_shots = int(((shots.location_x < 0) | (shots.location_x > PITCH_X) | (shots.location_y < 0) | (shots.location_y > PITCH_Y)).sum())
oob_ff = int(((ff.x < 0) | (ff.x > PITCH_X) | (ff.y < 0) | (ff.y > PITCH_Y)).sum())
log(STEP, "CHECK", "coordinates outside the 120 x 80 pitch", f"{oob_shots} shots, {oob_ff} freeze-frame rows",
    "Small overshoots are measurement noise; they are clipped to the pitch in steps 5 and 6.")

n_complete = int(summary.season_complete.astype(str).str.upper().eq("TRUE").sum())
print(f"League-seasons in summary: {len(summary)}; complete seasons: {n_complete}")

[06] 3. Checks | CHECK | duplicate shot_id: 0 duplicates
[07] 3. Checks | CHECK | missing values in shots: {'end_z': 15970}
[08] 3. Checks | CHECK | missing values in freeze_frames: none
[09] 3. Checks | CHECK | reconcile with summary.csv: 0 league-seasons disagree out of 27
[10] 3. Checks | CHECK | shots whose match_id is not in matches.csv: 0 match_ids
[11] 3. Checks | CHECK | freeze-frame consistency: 0 frame shot_ids not in shots; 0 shots flagged as having a frame but no rows; 0 shots with rows but flag False
[12] 3. Checks | CHECK | coordinates outside the 120 x 80 pitch: 2 shots, 0 freeze-frame rows
League-seasons in summary: 27; complete seasons: 3


## 4. Filter shots

In [4]:
STEP = "4. Filter shots"
shots_f = shots.copy()
n_start = len(shots_f)

# 4a. need a usable freeze frame
ok = shots_f.has_freeze_frame & shots_f.shot_id.isin(set(ff.shot_id))
n_drop = int((~ok).sum())
shots_f = shots_f[ok]
log(STEP, "REMOVE ROWS", "shots without a freeze frame", f"{n_drop:,} removed ({n_start:,} -> {len(shots_f):,})",
    "Both models take the freeze frame as input; a shot with no visible players cannot be used.")

# 4b. penalties
if EXCLUDE_PENALTIES:
    pen = shots_f.shot_type.eq("Penalty")
    log(STEP, "REMOVE ROWS", "penalties", f"{int(pen.sum())} removed; their goal rate was {shots_f.loc[pen, 'is_goal'].mean():.1%} vs {shots_f.loc[~pen, 'is_goal'].mean():.1%} for the rest",
        "Penalties are set pieces with a fixed geometry, so freeze-frame positions add little; they would also distort calibration. Can be modelled separately later.")
    shots_f = shots_f[~pen]

# 4c. exactly one opposition goalkeeper in the frame
opp_gk = ff.loc[(~ff.teammate) & ff.is_keeper].groupby("shot_id").size()
n_opp_gk = shots_f.shot_id.map(opp_gk).fillna(0).astype(int)
n0, n2 = int((n_opp_gk == 0).sum()), int((n_opp_gk >= 2).sum())
shots_f = shots_f[n_opp_gk.eq(1)]
log(STEP, "REMOVE ROWS", "frames without exactly one opposition goalkeeper",
    f"{n0} with none, {n2} with two or more removed -> {len(shots_f):,} shots remain",
    "The keeper position is one of the most important cues for xG; frames where it is missing or ambiguous would give the model a different input definition.")

shots_f = shots_f.sort_values("shot_id").reset_index(drop=True)
print("Goal rate after filtering:", round(shots_f.is_goal.mean(), 4))

[13] 4. Filter shots | REMOVE ROWS | shots without a freeze frame: 520 removed (54,153 -> 53,633)
[14] 4. Filter shots | REMOVE ROWS | penalties: 77 removed; their goal rate was 26.0% vs 10.3% for the rest
[15] 4. Filter shots | REMOVE ROWS | frames without exactly one opposition goalkeeper: 48 with none, 0 with two or more removed -> 53,508 shots remain
Goal rate after filtering: 0.1031


## 5. Shot table: select columns, clean, add geometry

In [5]:
STEP = "5. Shot table"

KEEP = ["shot_id", "match_id", "league", "season", "match_date", "season_complete",      # identifiers / metadata
        "location_x", "location_y", "shot_type", "body_part", "technique", "play_pattern",
        "first_time", "under_pressure",                                                   # model inputs
        "is_goal",                                                                        # target
        "statsbomb_xg"]                                                                   # benchmark only

DROP = {
    "minute": "Weak signal for xG and not needed for our question; removing reduces noise.",
    "second": "Same as minute.",
    "period": "Same as minute.",
    "team": "Team identity would let the network memorise teams (and does not scale / generalise).",
    "player": "Shooter identity: same memorisation problem. A 'shooter skill' extension is out of scope.",
    "outcome": "LEAKAGE: describes what happened after the shot (Goal / Saved / Post ...).",
    "end_x": "LEAKAGE: where the shot ended up is only known after the shot.",
    "end_y": "LEAKAGE: same as end_x.",
    "end_z": "LEAKAGE: same as end_x (also missing for ~30% of shots).",
    "has_freeze_frame": "Only used as a filter in step 4; constant True afterwards.",
    "ff_n_players": "Redundant: recomputed from the player table (n_visible_players).",
    "ff_n_teammates": "Redundant: derivable from the player table.",
    "ff_n_opponents": "Redundant: derivable from the player table.",
    "ff_has_keeper": "Redundant: every kept frame has exactly one opposition keeper (step 4).",
}
unknown = set(shots.columns) - set(KEEP) - set(DROP)
missing = (set(KEEP) | set(DROP)) - set(shots.columns)
assert not unknown and not missing, f"Column mismatch. Unaccounted: {unknown}; not in file: {missing}"

for c, why in DROP.items():
    log(STEP, "REMOVE COLUMN", f"shots.{c}", "dropped", why)
log(STEP, "KEEP COLUMN", "statsbomb_xg", "kept as benchmark column only", "Used as the 'ceiling' comparison in evaluation. It must NEVER be a model input.")
log(STEP, "KEEP COLUMN", "match_id, league, season, match_date, season_complete", "kept as metadata only",
    "Needed for the match-level split and for robustness checks (e.g. hold out a league). Not model inputs.")

shot_tbl = shots_f[KEEP].copy()

# clip coordinates to the pitch
for c, hi in [("location_x", PITCH_X), ("location_y", PITCH_Y)]:
    n_out = int(((shot_tbl[c] < 0) | (shot_tbl[c] > hi)).sum())
    shot_tbl[c] = shot_tbl[c].clip(0, hi)
    log(STEP, "TRANSFORM", f"{c}", f"{n_out} values clipped to [0, {hi:.0f}]", "Keeps every shooter on the pitch so distance / angle are well defined.")

# booleans -> 0/1
for c in ["first_time", "under_pressure", "is_goal"]:
    shot_tbl[c] = shot_tbl[c].astype(np.int8)
log(STEP, "TRANSFORM", "first_time, under_pressure, is_goal", "bool -> 0/1 int", "Neural networks need numeric inputs; is_goal is the target.")

# geometry features
shot_tbl["dist_goal"] = np.hypot(GOAL_X - shot_tbl.location_x, GOAL_Y - shot_tbl.location_y)
shot_tbl["angle_goal"] = subtended_angle(shot_tbl.location_x, shot_tbl.location_y)
log(STEP, "ADD COLUMNS", "dist_goal, angle_goal",
    "distance to goal centre (yards); angle between the two posts as seen from the shooter (radians)",
    "The two classic xG features. We state the goal geometry explicitly so the network does not have to learn trigonometry from raw coordinates.")
shot_tbl.describe().T[["count", "mean", "std", "min", "max"]]

[16] 5. Shot table | REMOVE COLUMN | shots.minute: dropped
[17] 5. Shot table | REMOVE COLUMN | shots.second: dropped
[18] 5. Shot table | REMOVE COLUMN | shots.period: dropped
[19] 5. Shot table | REMOVE COLUMN | shots.team: dropped
[20] 5. Shot table | REMOVE COLUMN | shots.player: dropped
[21] 5. Shot table | REMOVE COLUMN | shots.outcome: dropped
[22] 5. Shot table | REMOVE COLUMN | shots.end_x: dropped
[23] 5. Shot table | REMOVE COLUMN | shots.end_y: dropped
[24] 5. Shot table | REMOVE COLUMN | shots.end_z: dropped
[25] 5. Shot table | REMOVE COLUMN | shots.has_freeze_frame: dropped
[26] 5. Shot table | REMOVE COLUMN | shots.ff_n_players: dropped
[27] 5. Shot table | REMOVE COLUMN | shots.ff_n_teammates: dropped
[28] 5. Shot table | REMOVE COLUMN | shots.ff_n_opponents: dropped
[29] 5. Shot table | REMOVE COLUMN | shots.ff_has_keeper: dropped
[30] 5. Shot table | KEEP COLUMN | statsbomb_xg: kept as benchmark column only
[31] 5. Shot table | KEEP COLUMN | match_id, league, season,

,count,mean,std,min,max
match_id,53508.0,3.015015e+06,1.538236e+06,9575.00000,3.901263e+06
location_x,53508.0,1.033315e+02,8.813557e+00,35.40000,1.200000e+02
location_y,53508.0,3.962687e+01,1.002412e+01,0.30000,7.960000e+01
first_time,53508.0,3.147380e-01,4.644158e-01,0.00000,1.000000e+00
under_pressure,53508.0,2.372916e-01,4.254265e-01,0.00000,1.000000e+00
is_goal,53508.0,1.031435e-01,3.041490e-01,0.00000,1.000000e+00
statsbomb_xg,53508.0,9.546567e-02,1.249757e-01,0.00018,9.951224e-01
dist_goal,53508.0,1.950315e+01,8.704369e+00,0.40000,8.883316e+01
angle_goal,53508.0,4.316505e-01,2.666143e-01,0.00000,3.141593e+00


## 6. Player table: freeze frames + shooter row + geometry

In [6]:
STEP = "6. Player table"

ff_f = ff[ff.shot_id.isin(set(shot_tbl.shot_id))].copy()
log(STEP, "REMOVE ROWS", "freeze-frame rows of removed shots", f"{len(ff) - len(ff_f):,} removed ({len(ff):,} -> {len(ff_f):,})",
    "Keep the two tables consistent: every player row belongs to a shot in the shot table.")

log(STEP, "REMOVE COLUMN", "freeze_frames.player_name", "dropped", "Identity -> memorisation, high cardinality, does not scale.")
def position_group(p):
    if "Goalkeeper" in p: return "GK"
    if "Back" in p: return "DEF"          # Center Back, Left Back, Wing Back ...
    if "Midfield" in p: return "MID"
    return "FWD"                           # Wing, Forward, Striker
ff_f["position_group"] = ff_f.position.map(position_group)
ff_f = ff_f.drop(columns=["player_name", "position"])
log(STEP, "ADD COLUMNS", "position_group", "25 StatsBomb positions collapsed to GK / DEF / MID / FWD (kept in the table, NOT in the default feature list)",
    "Optional ablation: role information may help, but x, y and the team / keeper flags already capture most of it.")

for c, hi in [("x", PITCH_X), ("y", PITCH_Y)]:
    n_out = int(((ff_f[c] < 0) | (ff_f[c] > hi)).sum())
    ff_f[c] = ff_f[c].clip(0, hi)
    log(STEP, "TRANSFORM", f"freeze_frames.{c}", f"{n_out} values clipped", "Same pitch bounds as the shooter.")

ff_f["teammate"] = ff_f.teammate.astype(np.int8)
ff_f["is_keeper"] = ff_f.is_keeper.astype(np.int8)
ff_f["is_shooter"] = np.int8(0)

# add the shooter as an extra row (their position is stored in the shot table, not in the freeze frame)
sh = shot_tbl[["shot_id", "location_x", "location_y"]].rename(columns={"location_x": "x", "location_y": "y"})
sh["teammate"] = np.int8(1)
sh["is_keeper"] = np.int8(0)
sh["is_shooter"] = np.int8(1)
sh["position_group"] = "SHOOTER"
players = pd.concat([sh, ff_f], ignore_index=True)
players = players.sort_values(["shot_id", "is_shooter"], ascending=[True, False], kind="stable").reset_index(drop=True)
players["node_idx"] = players.groupby("shot_id").cumcount()          # 0 = shooter
log(STEP, "ADD ROWS", "shooter row for every shot", f"{len(sh):,} rows added",
    "The shooter is not in the freeze frame; we add them as a node (is_shooter = 1, teammate = 1) so the model knows where the shot is taken from.")

# geometry relative to the shooter
sxy = shot_tbl.set_index("shot_id")
sx = players.shot_id.map(sxy.location_x).to_numpy()
sy = players.shot_id.map(sxy.location_y).to_numpy()
px, py = players.x.to_numpy(), players.y.to_numpy()
is_sh = players.is_shooter.to_numpy().astype(bool)

players["dx"] = px - sx
players["dy"] = py - sy
players["dist_goal"] = np.hypot(GOAL_X - px, GOAL_Y - py)
players["angle_goal"] = subtended_angle(px, py)
players["dist_shooter"] = np.hypot(players.dx, players.dy)
ang = np.arctan2(players.dy, players.dx)                              # 0 = straight towards the goal line
players["angle_shooter"] = np.where(is_sh, 0.0, ang)
players["angle_shooter_sin"] = np.where(is_sh, 0.0, np.sin(ang))
players["angle_shooter_cos"] = np.where(is_sh, 0.0, np.cos(ang))
players["dist_to_shot_line"] = np.where(is_sh, 0.0, dist_to_segment(px, py, sx, sy, GOAL_X, GOAL_Y))
players["in_shot_triangle"] = (in_triangle(px, py, sx, sy) & ~is_sh).astype(np.int8)

log(STEP, "ADD COLUMNS", "dx, dy, dist_shooter, angle_shooter", "position relative to the shooter (yards / radians)",
    "Positions relative to the shooter are what matters for blocking; also needed later to build graph edges (7-yard / 3-4-yard regions).")
log(STEP, "ADD COLUMNS", "angle_shooter_sin, angle_shooter_cos", "sin / cos of angle_shooter (0 for the shooter row)",
    "A raw angle jumps from +pi to -pi directly behind the shooter; sin / cos are continuous, so they are used as the model inputs.")
log(STEP, "ADD COLUMNS", "dist_goal, angle_goal (per player)", "same geometry as in the shot table",
    "Tells the model how close each player is to goal and how much of the goal they cover.")
log(STEP, "ADD COLUMNS", "dist_to_shot_line, in_shot_triangle", "distance to the shooter->goal-centre line; flag for being inside the shooter-posts triangle",
    "Direct measures of 'is this player blocking the shot?' so the network does not have to discover them.")

vis = players.loc[~players.is_shooter.astype(bool)].groupby("shot_id").size()
shot_tbl["n_visible_players"] = shot_tbl.shot_id.map(vis).astype(np.int16)
log(STEP, "ADD COLUMNS", "shot_table.n_visible_players", "number of visible players (excluding the shooter)",
    "Metadata for padding / batching and sanity checks; not a model input (the player table already encodes it).")
players.head(8)

[36] 6. Player table | REMOVE ROWS | freeze-frame rows of removed shots: 613 removed (685,059 -> 684,446)
[37] 6. Player table | REMOVE COLUMN | freeze_frames.player_name: dropped
[38] 6. Player table | ADD COLUMNS | position_group: 25 StatsBomb positions collapsed to GK / DEF / MID / FWD (kept in the table, NOT in the default feature list)
[39] 6. Player table | TRANSFORM | freeze_frames.x: 0 values clipped
[40] 6. Player table | TRANSFORM | freeze_frames.y: 0 values clipped
[41] 6. Player table | ADD ROWS | shooter row for every shot: 53,508 rows added
[42] 6. Player table | ADD COLUMNS | dx, dy, dist_shooter, angle_shooter: position relative to the shooter (yards / radians)
[43] 6. Player table | ADD COLUMNS | angle_shooter_sin, angle_shooter_cos: sin / cos of angle_shooter (0 for the shooter row)
[44] 6. Player table | ADD COLUMNS | dist_goal, angle_goal (per player): same geometry as in the shot table
[45] 6. Player table | ADD COLUMNS | dist_to_shot_line, in_shot_triangle: distan

,shot_id,x,y,teammate,is_keeper,is_shooter,position_group,node_idx,dx,dy,dist_goal,angle_goal,dist_shooter,angle_shooter,angle_shooter_sin,angle_shooter_cos,dist_to_shot_line,in_shot_triangle
0,00007fd2-738e-4b27-bb68-7bdeffc77ee7,101.2,24.0,1,0,1,SHOOTER,0,0.0,0.0,24.686839,0.248211,0.000000,0.000000,0.000000,0.000000,0.000000,0
1,00007fd2-738e-4b27-bb68-7bdeffc77ee7,104.2,43.1,0,0,0,DEF,1,3.0,19.1,16.101242,0.479228,19.334167,1.415001,0.987888,0.155166,12.601046,0
2,00007fd2-738e-4b27-bb68-7bdeffc77ee7,94.4,30.8,0,0,0,MID,2,-6.8,6.8,27.202941,0.275671,9.616652,2.356194,0.707107,-0.707107,9.616652,0
3,00007fd2-738e-4b27-bb68-7bdeffc77ee7,97.4,45.0,0,0,0,MID,3,-3.8,21.0,23.146490,0.334761,21.341040,1.749812,0.984020,-0.178061,18.455178,0
4,00007fd2-738e-4b27-bb68-7bdeffc77ee7,102.1,23.8,0,0,0,DEF,4,0.9,-0.2,24.142287,0.247447,0.921954,-0.218669,-0.216930,0.976187,0.735615,0
5,00007fd2-738e-4b27-bb68-7bdeffc77ee7,103.3,48.2,0,0,0,DEF,5,2.1,24.2,18.604569,0.384542,24.290945,1.484236,0.996256,0.086452,17.068204,0
6,00007fd2-738e-4b27-bb68-7bdeffc77ee7,99.6,56.9,1,0,0,MID,6,-1.6,32.9,26.490942,0.233635,32.938883,1.619390,0.998820,-0.048575,26.091636,0
7,00007fd2-738e-4b27-bb68-7bdeffc77ee7,93.3,45.7,1,0,0,FWD,7,-7.9,21.7,27.301648,0.284886,23.093289,1.919937,0.939667,-0.342091,21.645542,0


## 7. Train / validation / test split (by match)

In [7]:
STEP = "7. Split"
rng = np.random.default_rng(RANDOM_SEED)
match_ids = np.sort(shot_tbl.match_id.unique())
rng.shuffle(match_ids)
n = len(match_ids)
n_tr = int(round(n * SPLIT_FRACS["train"]))
n_va = int(round(n * SPLIT_FRACS["val"]))
split_of = {}
for i, m in enumerate(match_ids):
    split_of[m] = "train" if i < n_tr else ("val" if i < n_tr + n_va else "test")

shot_tbl["split"] = shot_tbl.match_id.map(split_of)
players["split"] = players.shot_id.map(shot_tbl.set_index("shot_id").split)

summ = shot_tbl.groupby("split").agg(matches=("match_id", "nunique"), shots=("shot_id", "size"), goal_rate=("is_goal", "mean"))
log(STEP, "SPLIT", "70 / 15 / 15 by match_id", summ.round(4).to_dict("index").__repr__(),
    "Shots from the same match share teams, players and conditions, so a random shot-level split would leak information. Splitting by match avoids this.")
summ

[47] 7. Split | SPLIT | 70 / 15 / 15 by match_id: {'test': {'matches': 326, 'shots': 8075, 'goal_rate': 0.0994}, 'train': {'matches': 1518, 'shots': 37431, 'goal_rate': 0.1026}, 'val': {'matches': 325, 'shots': 8002, 'goal_rate': 0.1092}}


,matches,shots,goal_rate
split,,,
test,326,8075,0.099443
train,1518,37431,0.102642
val,325,8002,0.109223


## 8. One-hot encoding (shot table)

In [8]:
STEP = "8. Encode"
CATS = ["shot_type", "body_part", "technique", "play_pattern"]
train_mask = shot_tbl.split.eq("train")

onehot_cols = []
for col in CATS:
    counts = shot_tbl.loc[train_mask, col].value_counts()
    common = set(counts[counts >= RARE_MIN].index)
    rare = sorted(set(shot_tbl[col].unique()) - common)
    if rare:
        shot_tbl[col] = shot_tbl[col].where(shot_tbl[col].isin(common), "Other")
        log(STEP, "TRANSFORM", f"{col}", f"categories grouped into 'Other': {rare}",
            f"Fewer than {RARE_MIN} training shots: too rare for the network to learn anything reliable (decided from the train split only).")
    d = pd.get_dummies(shot_tbl[col], prefix=col, dtype=np.int8)
    d.columns = [re.sub(r"\W+", "_", c).strip("_") for c in d.columns]
    shot_tbl = pd.concat([shot_tbl.drop(columns=col), d], axis=1)
    onehot_cols += list(d.columns)
    log(STEP, "ENCODE", f"{col}", f"one-hot -> {list(d.columns)}",
        "Categorical text cannot be fed to a network. All categories are kept (no 'drop first') so the encoding is symmetric for the network.")
log(STEP, "REMOVE COLUMN", "original text columns shot_type, body_part, technique, play_pattern", "dropped after encoding",
    "Prevents accidentally feeding raw text to the model; the information is fully contained in the one-hot columns.")
print(len(onehot_cols), "one-hot columns")

[48] 8. Encode | TRANSFORM | shot_type: categories grouped into 'Other': ['Corner']
[49] 8. Encode | ENCODE | shot_type: one-hot -> ['shot_type_Free_Kick', 'shot_type_Open_Play', 'shot_type_Other']
[50] 8. Encode | ENCODE | body_part: one-hot -> ['body_part_Head', 'body_part_Left_Foot', 'body_part_Other', 'body_part_Right_Foot']
[51] 8. Encode | ENCODE | technique: one-hot -> ['technique_Backheel', 'technique_Diving_Header', 'technique_Half_Volley', 'technique_Lob', 'technique_Normal', 'technique_Overhead_Kick', 'technique_Volley']
[52] 8. Encode | ENCODE | play_pattern: one-hot -> ['play_pattern_From_Corner', 'play_pattern_From_Counter', 'play_pattern_From_Free_Kick', 'play_pattern_From_Goal_Kick', 'play_pattern_From_Keeper', 'play_pattern_From_Kick_Off', 'play_pattern_From_Throw_In', 'play_pattern_Other', 'play_pattern_Regular_Play']
[53] 8. Encode | REMOVE COLUMN | original text columns shot_type, body_part, technique, play_pattern: dropped after encoding
23 one-hot columns


## 9. Standardisation (statistics from the training split only)

In [9]:
STEP = "9. Scale"
def fit_scaler(df, cols, mask):
    stats = {}
    for c in cols:
        mu = float(df.loc[mask, c].mean())
        sd = float(df.loc[mask, c].std(ddof=0))
        stats[c] = {"mean": mu, "std": sd if sd > 1e-9 else 1.0}
    return stats

def apply_scaler(df, stats):
    for c, s in stats.items():
        df[c + "_z"] = ((df[c] - s["mean"]) / s["std"]).astype(np.float32)

SHOT_NUM = ["location_x", "location_y", "dist_goal", "angle_goal"]
PLAYER_NUM = ["x", "y", "dx", "dy", "dist_goal", "angle_goal", "dist_shooter", "dist_to_shot_line"]

scaler = {"shot": fit_scaler(shot_tbl, SHOT_NUM, shot_tbl.split.eq("train")),
          "player": fit_scaler(players, PLAYER_NUM, players.split.eq("train"))}
apply_scaler(shot_tbl, scaler["shot"])
apply_scaler(players, scaler["player"])

log(STEP, "SCALE", "shot numeric features", f"standardised (z-score): {SHOT_NUM} -> *_z",
    "Inputs on very different scales (yards vs radians) slow down and destabilise training. Mean / std come from the TRAIN split only, so no information from val / test leaks in.")
log(STEP, "SCALE", "player numeric features", f"standardised (z-score): {PLAYER_NUM} -> *_z",
    "Same reason. Binary flags and the sin / cos columns are not scaled (already in [-1, 1]).")
log(STEP, "KEEP COLUMN", "unscaled yard / radian columns", "kept next to the *_z columns",
    "Graph edges are defined in real yards (e.g. 7-yard radius), so the raw geometry is needed when building the graph. Models should use the *_z columns.")

[54] 9. Scale | SCALE | shot numeric features: standardised (z-score): ['location_x', 'location_y', 'dist_goal', 'angle_goal'] -> *_z
[55] 9. Scale | SCALE | player numeric features: standardised (z-score): ['x', 'y', 'dx', 'dy', 'dist_goal', 'angle_goal', 'dist_shooter', 'dist_to_shot_line'] -> *_z
[56] 9. Scale | KEEP COLUMN | unscaled yard / radian columns: kept next to the *_z columns


## 10. Final columns and feature lists

In [10]:
STEP = "10. Columns"
ID_COLS = ["shot_id", "match_id"]
META_COLS = ["league", "season", "match_date", "season_complete", "split", "n_visible_players"]
TARGET = "is_goal"
BENCHMARK = ["statsbomb_xg"]

SHOT_FEATURES = ["location_x_z", "location_y_z", "dist_goal_z", "angle_goal_z", "first_time", "under_pressure"] + onehot_cols
PLAYER_FEATURES = ["x_z", "y_z", "dx_z", "dy_z", "dist_goal_z", "angle_goal_z", "dist_shooter_z", "dist_to_shot_line_z",
                   "angle_shooter_sin", "angle_shooter_cos", "teammate", "is_keeper", "is_shooter", "in_shot_triangle"]
RAW_SHOT_GEOM = ["location_x", "location_y", "dist_goal", "angle_goal"]
RAW_PLAYER_GEOM = ["x", "y", "dx", "dy", "dist_goal", "angle_goal", "dist_shooter", "angle_shooter", "dist_to_shot_line"]

shot_out = shot_tbl[ID_COLS + META_COLS + RAW_SHOT_GEOM + SHOT_FEATURES + [TARGET] + BENCHMARK].reset_index(drop=True)

player_out = players[["shot_id", "node_idx", "split", "position_group"] + RAW_PLAYER_GEOM + PLAYER_FEATURES].reset_index(drop=True)

feature_spec = {"id_cols": ID_COLS, "meta_cols": META_COLS, "target": TARGET, "benchmark_only": BENCHMARK,
                "shot_features": SHOT_FEATURES, "player_features": PLAYER_FEATURES,
                "raw_shot_geometry": RAW_SHOT_GEOM, "raw_player_geometry": RAW_PLAYER_GEOM,
                "optional_player_columns": ["position_group"]}
log(STEP, "FEATURE LIST", "shot model inputs", f"{len(SHOT_FEATURES)} columns", "Saved in feature_columns.json so every notebook uses the same inputs.")
log(STEP, "FEATURE LIST", "player model inputs", f"{len(PLAYER_FEATURES)} columns", "Same; the set and graph models both use these node features.")
print("shot table  :", shot_out.shape)
print("player table:", player_out.shape)

[57] 10. Columns | FEATURE LIST | shot model inputs: 29 columns
[58] 10. Columns | FEATURE LIST | player model inputs: 14 columns
shot table  : (53508, 43)
player table: (737954, 27)


## 11. Validation checks

In [11]:
STEP = "11. Validate"
# 1. no NaN / inf in model inputs
assert not shot_out[SHOT_FEATURES + [TARGET]].isna().any().any(), "NaN in shot features"
assert not player_out[PLAYER_FEATURES].isna().any().any(), "NaN in player features"
assert np.isfinite(shot_out[SHOT_FEATURES].to_numpy(dtype=float)).all()
assert np.isfinite(player_out[PLAYER_FEATURES].to_numpy(dtype=float)).all()
# 2. tables link perfectly
assert set(shot_out.shot_id) == set(player_out.shot_id)
assert shot_out.shot_id.is_unique
# 3. exactly one shooter and one opposition keeper per shot
g = player_out.groupby("shot_id")
assert (g.is_shooter.sum() == 1).all()
assert (player_out.assign(opp_gk=(player_out.is_keeper == 1) & (player_out.teammate == 0)).groupby("shot_id").opp_gk.sum() == 1).all()
# 4. row counts add up: visible players + one shooter row per shot
assert len(player_out) == shot_out.n_visible_players.sum() + len(shot_out)
# 5. match-level split has no overlap
sets = {s: set(shot_out.loc[shot_out.split == s, "match_id"]) for s in ["train", "val", "test"]}
assert not (sets["train"] & sets["val"]) and not (sets["train"] & sets["test"]) and not (sets["val"] & sets["test"])
# 6. train columns standardised
zc = [c for c in shot_out.columns if c.endswith("_z")]
tr = shot_out[shot_out.split == "train"]
assert np.allclose(tr[zc].mean(), 0, atol=1e-3) and np.allclose(tr[zc].std(ddof=0), 1, atol=1e-2)
# 7. nothing leaky is in the input lists
assert not (set(SHOT_FEATURES) & {"statsbomb_xg", "outcome", "end_x", "end_y", "end_z", "is_goal"})

log(STEP, "CHECK", "all validation asserts", "passed",
    "No NaN / inf; tables link on shot_id; exactly one shooter and one opposition keeper per shot; row counts add up; no match is in two splits; train z-scores have mean 0 / std 1; no leaky column in the inputs.")
print(shot_out.groupby("split").agg(shots=("shot_id", "size"), goal_rate=("is_goal", "mean"), mean_visible=("n_visible_players", "mean")).round(3))
print("\nPlayers per shot (visible, excluding shooter):")
print(shot_out.n_visible_players.describe().round(2))

[59] 11. Validate | CHECK | all validation asserts: passed
       shots  goal_rate  mean_visible
split                                
test    8075      0.099        12.854
train  37431      0.103        12.757
val     8002      0.109        12.889

Players per shot (visible, excluding shooter):
count    53508.00
mean        12.79
std          3.58
min          2.00
25%         10.00
50%         13.00
75%         16.00
max         21.00
Name: n_visible_players, dtype: float64


## 12. Save outputs

In [12]:
STEP = "12. Save"
SHOT_CSV = PROC_DIR / "shot_table.csv"
PLAYER_CSV = PROC_DIR / "player_table.csv"
shot_out.to_csv(SHOT_CSV, index=False, float_format="%.6g")
player_out.to_csv(PLAYER_CSV, index=False, float_format="%.6g")
log(STEP, "SAVE", "shot_table.csv", f"{shot_out.shape[0]:,} rows x {shot_out.shape[1]} cols -> {SHOT_CSV.name}", "Model-ready shot table (1 row per shot).")
log(STEP, "SAVE", "player_table.csv", f"{player_out.shape[0]:,} rows x {player_out.shape[1]} cols -> {PLAYER_CSV.name}", "Model-ready player table (1 row per player per shot, shooter included).")

try:
    shot_out.to_parquet(PROC_DIR / "shot_table.parquet", index=False)
    player_out.to_parquet(PROC_DIR / "player_table.parquet", index=False)
    log(STEP, "SAVE", "parquet copies", "written", "Columnar and compressed: much faster to load and the natural format for the 1000x scaling argument.")
except Exception as e:
    log(STEP, "SAVE", "parquet copies", f"skipped ({type(e).__name__})", "pyarrow is not installed; CSVs are enough. `pip install pyarrow` to enable.")

with open(PROC_DIR / "scaler_params.json", "w") as f:
    json.dump(scaler, f, indent=2)
with open(PROC_DIR / "feature_columns.json", "w") as f:
    json.dump(feature_spec, f, indent=2)
log(STEP, "SAVE", "scaler_params.json, feature_columns.json", "written",
    "Mean / std from the train split (to scale any new data identically) and the list of model input columns.")

[60] 12. Save | SAVE | shot_table.csv: 53,508 rows x 43 cols -> shot_table.csv
[61] 12. Save | SAVE | player_table.csv: 737,954 rows x 27 cols -> player_table.csv
[62] 12. Save | SAVE | parquet copies: written
[63] 12. Save | SAVE | scaler_params.json, feature_columns.json: written


## 13. Processing log

In [13]:
log_df = pd.DataFrame(LOG)
log_df.to_csv(PROC_DIR / "processing_log.csv", index=False)

lines = ["# Processing log (01_data_processing.ipynb)", ""]
for step, grp in log_df.groupby("step", sort=False):
    lines += [f"## {step}", ""]
    for _, r in grp.iterrows():
        lines.append(f"- **{r.action}** - {r['item']}: {r.detail}  \n  *Why:* {r.reason}")
    lines.append("")
(PROC_DIR / "processing_log.md").write_text("\n".join(lines), encoding="utf-8")

print(f"{len(log_df)} log entries saved to {PROC_DIR}")
pd.set_option("display.max_colwidth", 140)
log_df[["n", "step", "action", "item", "detail", "reason"]]

63 log entries saved to /home/jovyan/Data Science Toolbox/Group Sprint 2/data/processed


,n,step,action,item,detail,reason
0,1,2. Load,LOAD,shots.csv,"54,153 rows x 30 cols",Main shot-level data (one row per shot).
1,2,2. Load,LOAD,freeze_frames.csv,"685,059 rows x 7 cols",One row per visible player per shot.
2,3,2. Load,LOAD,matches.csv,"2,169 rows",Used only for integrity checks and context; not a model input.
3,4,2. Load,LOAD,summary.csv,27 league-seasons,Used only to reconcile our counts with the source; not a model input.
4,5,2. Load,TRANSFORM,boolean columns,TRUE/FALSE text parsed to bool,Make sure flags are real booleans before converting to 0/1 later.
...,...,...,...,...,...,...
58,59,11. Validate,CHECK,all validation asserts,passed,No NaN / inf; tables link on shot_id; exactly one shooter and one opposition keeper per shot; row counts add up; no match is in two spli...
59,60,12. Save,SAVE,shot_table.csv,"53,508 rows x 43 cols -> shot_table.csv",Model-ready shot table (1 row per shot).
60,61,12. Save,SAVE,player_table.csv,"737,954 rows x 27 cols -> player_table.csv","Model-ready player table (1 row per player per shot, shooter included)."
61,62,12. Save,SAVE,parquet copies,written,Columnar and compressed: much faster to load and the natural format for the 1000x scaling argument.


## Decisions deliberately left for the modelling notebooks

- **Graph edges** (shooter regions, shooter-only vs. non-shooter edges, goal / post nodes): built in the modelling notebook from the raw yard columns (`dx`, `dy`, `dist_shooter`, `angle_shooter`). Radii are hyperparameters to tune on the validation split.
- **Mirror augmentation** (reflect y -> 80 - y, swap left / right foot): not applied here.
- **Penalties**: removed from the main tables (`EXCLUDE_PENALTIES`); can be modelled separately.
- **Optional features**: `position_group` is in the player table but not in the default input list.
- **Hold-out league / season tests**: `league`, `season` and `season_complete` are kept as metadata for this.